# 03 预训练、冻结与微调：参数从哪里开始，哪些继续学习

你已经知道标签怎样表示真实框、训练集与验证集怎样分工。这一课接着回答：**开始训练时，网络参数从哪里来？训练时，又允许哪些参数改变？**

先记住两个独立的选择：

- 加载哪份权重，决定参数的起点。
- 冻结哪些层，决定哪些参数允许继续学习。

本课先讲原理，再对照本地 YOLOv5 的实际加载逻辑，不运行训练。

## 1. 从头训练与加载预训练权重

| 起点 | 参数最初的状态 | 本次训练要做什么 |
|---|---|---|
| 从头训练 | 主要是随机初始化，部分参数按结构规则初始化 | 从当前数据中学习视觉特征和任务规律 |
| 加载预训练权重 | 可兼容的参数继承已有训练结果 | 在已有特征基础上适应当前任务 |
| 加载已有口罩权重 | 已经学习过口罩检测任务 | 继续适应新增数据、目标场景或训练目标 |

预训练权重不是一份“答案图片库”，而是网络已经学到的参数。加载它，不等于网络必然适应新场景。

已学过的边缘、纹理等特征，可能帮助新任务起步。但实际效果仍取决于预训练任务、数据分布、网络结构和后续训练方式。

## 2. 冻结不等于停止计算

回顾刚才的讨论，用一个简单函数理解：

$$
y=ax+b
$$

固定 $a=2,b=1$，输入 $x=1$ 时输出 3，输入 $x=2$ 时输出 5。

固定的是参数，不是输入，更不是输出。

同样，冻结骨干网络后：

| 行为 | 是否继续 |
|---|---|
| 接收图片并进行前向计算 | 是 |
| 根据不同图片提取不同特征 | 仍然可能 |
| 更新被冻结的权重参数 | 否 |

你刚才已经正确指出：戴口罩和不戴口罩的图片输入同一个冻结骨干，特征不一定相同。

对简单串联的骨干和检测头而言，如果骨干及输入都不需要梯度，可以不为骨干建立求导图，但它的前向计算仍然必须完成。

## 3. 两种常见的微调方式

把网络粗略看成“骨干提取特征，检测头利用特征做预测”。真实 YOLO 还包含 Neck，这里先用简化结构理解参数学习。

| 方式 | 继承的骨干参数 | 检测头参数 | 含义 |
|---|---|---|---|
| 冻结骨干、训练检测头 | 保持不变 | 允许更新 | 学习怎样使用已有特征 |
| 整个网络微调 | 允许更新 | 允许更新 | 特征提取方法与检测预测共同适应任务 |

冻结的好处是减少需要学习的参数及相关训练开销；局限是骨干无法通过权重更新来适应新任务。

整个网络微调更灵活，但仍要控制更新幅度并观察验证结果，不能保证一定优于冻结训练。

“冻结骨干、训练头”不代表任何代码中的冻结选项都只留下最后一层可训练。具体哪些模块被冻结，要对照当前模型结构核对。

## 4. 通用类别与口罩类别不同，怎么办

设一份通用 COCO 检测权重学习的是 person、car 等 80 类；当前口罩数据需要的是另外三类：

| 编号 | 当前任务含义 |
|---|---|
| 0 | with_mask |
| 1 | without_mask |
| 2 | mask_weared_incorrect |

**把 person 的显示文字改成 with_mask，并不会让模型学会口罩检测。** 类别名字与参数学到的判别规律是两回事。

同时，预测输出的尺寸也与类别数有关。将 80 类任务改为 3 类任务时，部分输出层的参数形状可能不再匹配，不能原样搬过去。

可以理解为：继承能够兼容的视觉特征，让任务相关的新参数通过口罩标注学习。

任务变化可能还涉及标注对象本身：通用 person 类通常标注整个人，而当前数据标注的是人脸区域。新的框位置也需要通过训练适应。

本节是假设从通用权重迁移的例子；当前网页实际使用的是作者的三类口罩权重，不是正在运行 80 类通用模型。

## 5. 本地 YOLOv5 实际怎样处理

阅读当前 `vendor/yolov5/train.py`，加载已有权重时的主要逻辑是：

1. 从数据配置中读取类别数和类别名字。
2. 根据模型配置创建适合当前类别数的模型。
3. 从旧权重中筛选名称、形状等条件相兼容的参数。
4. 加载匹配的参数，未加载的部分保留初始化值。
5. 根据冻结设置决定哪些参数允许求梯度，随后建立训练流程。

因此，加载预训练权重通常不代表每个参数都成功继承。训练日志中的 `Transferred ... items` 可以帮助检查实际迁移情况。

当前代码的默认冻结设置不冻结模型参数；也就是说，默认加载预训练权重以后，并不是自动只训练检测头。

这里描述的是本地固定版本的代码行为，后续实际训练时仍要核对启动参数和日志。

## 6. 学习率控制参数每次改多少

你已经学过最简单的 SGD 更新规则：

$$
\theta_{t+1}=\theta_t-\eta\nabla_\theta L
$$

其中 $\theta$ 表示参数，$\eta$ 是学习率，$L$ 是损失。

- 加载预训练权重：改变起始参数 $\theta_0$。
- 冻结参数：让指定参数保持不变，不参与相应的优化更新。
- 调整学习率：控制允许更新的参数每步改变的幅度。

它们是不同的选择。微调仍然是训练，仍然需要前向传播、损失、反向传播和优化器更新。

已有参数可能已经有用，更新过猛会破坏已有能力；但“学习率越小越好”也不成立，太小可能使适应任务的过程很慢。具体数值留到理解训练配置时再决定。

## 7. 回到当前口罩项目

我们目前有作者提供的 `models/mask_yolov5.pt`。网页加载后进行推理，并没有更新模型参数。

| 操作 | 是否构成微调 |
|---|---|
| 加载权重，检测一张图片 | 否 |
| 调整置信度阈值，让显示的框变少 | 否 |
| 只修改类别显示文字 | 否 |
| 利用标注计算损失并更新允许训练的参数 | 是，若从已有权重开始，就是微调 |

后续可以沿两条路线学习：

- 从通用检测预训练权重开始，学习让模型适应三类口罩任务。
- 从作者的口罩权重开始，用新场景数据学习进一步适应。

两条路线都要结合独立验证来判断效果。作者权重可能已经见过我们下载的数据，不能通过重新随机划分就默认消除这种影响。

本课没有下载新权重、选择具体训练超参数或启动训练。

## 8. 一个容易混淆的实现细节

“冻结权重参数”与“设置评估模式”不是同一个操作。

本地训练代码通过参数的 `requires_grad` 控制冻结；这不会自动保证 BatchNorm 的运行均值、运行方差等状态也停止变化。那些状态并不是普通的可训练权重，行为还与模块的训练或评估模式有关。

现在先记住：**冻结哪些参数、模块处于什么模式，是两个需要分别确认的问题。** 正式配置时再核对，无需现在展开 BatchNorm 的全部细节。

## 9. 自检：把三个选择分开

1. 骨干冻结后，为什么还能区分不同输入图片的特征？
2. 已加载预训练权重，是否意味着只能训练检测头？
3. 把通用模型的类别名字改成口罩类别，是否就完成了迁移学习？
4. 加载权重、冻结层、调整学习率，分别决定了什么？

<details>
<summary>参考答案</summary>

1. 参数保持不变，但仍对输入执行前向计算；不同输入仍可能得到不同输出。
2. 不意味着。是否更新骨干参数由冻结和优化设置决定。
3. 没有。名字只是类别映射，新的任务判别和定位能力需要相应训练。
4. 分别决定参数起点、哪些参数允许继续学习、每步更新幅度。

</details>

下一课：用一张图片串起 YOLO 的训练过程，理解模型预测、真实标签和损失之间的关系。

## 本课代码依据

- `vendor/yolov5/train.py`：读取数据类别数、创建模型、筛选与加载可迁移参数、设置冻结参数。
- `vendor/yolov5/data/coco.yaml`：通用 COCO 类别列表。
- `configs/mask.yaml`：本地三类口罩数据配置。
- `mask_app/detector.py`：当前界面的推理流程。

以上路径均相对于本地 `yolov5-face-mask-detection` 项目。YOLOv5 固定源码版本为 `af08ba156fd58ed5b0fe7e4e58a22c72c3cdd0ce`。

本次仅更新学习笔记，没有修改部署代码、数据或权重。